<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-33.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 33 - Calidad de Datos

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivo

En este notebook implementaremos un **pipeline completo de evaluación de calidad de datos**:

1. Generar un dataset "sucio" con problemas reales
2. Detectar nulos, duplicados, outliers e inconsistencias
3. Calcular un **quality score** global
4. Limpiar los datos y verificar la mejora

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Librerías cargadas correctamente ✅")

---

## 1. Generación del dataset "sucio"

Vamos a crear un dataset de clientes con **problemas de calidad intencionados** que simulan situaciones reales.

In [ ]:
N = 5000

nombres = ['Ana García', 'Carlos López', 'María Rodríguez', 'Pedro Martínez',
           'Laura Sánchez', 'David Fernández', 'Elena Torres', 'Miguel Ruiz',
           'Sofía Moreno', 'Javier Díaz', 'Paula Jiménez', 'Andrés Romero']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Málaga']

df = pd.DataFrame({
    'id_cliente': range(1, N + 1),
    'nombre': np.random.choice(nombres, N),
    'edad': np.random.randint(18, 70, N).astype(float),
    'email': [f'usuario{i}@email.com' for i in range(1, N + 1)],
    'ciudad': np.random.choice(ciudades, N),
    'ingresos': np.round(np.random.normal(35000, 12000, N), 2),
    'compras_anual': np.random.randint(0, 50, N).astype(float),
    'satisfaccion': np.random.randint(1, 11, N).astype(float),
    'fecha_registro': pd.date_range('2020-01-01', periods=N, freq='2h').astype(str)
})

# --- INYECTAR PROBLEMAS DE CALIDAD ---

# 1) Valores nulos aleatorios (~8% en varias columnas)
for col in ['edad', 'email', 'ingresos', 'compras_anual', 'satisfaccion']:
    mask = np.random.random(N) < 0.08
    df.loc[mask, col] = np.nan

# 2) Duplicados exactos (~3% de filas)
n_duplicados = int(N * 0.03)
filas_duplicadas = df.sample(n_duplicados, random_state=42)
df = pd.concat([df, filas_duplicadas], ignore_index=True)

# 3) Outliers en ingresos (valores extremos)
outlier_idx = np.random.choice(len(df), 25, replace=False)
df.loc[outlier_idx, 'ingresos'] = np.random.choice([500000, -5000, 900000, -10000], 25)

# 4) Edades inválidas
edad_invalida_idx = np.random.choice(len(df), 30, replace=False)
df.loc[edad_invalida_idx, 'edad'] = np.random.choice([-5, 0, 150, 200, 999], 30)

# 5) Inconsistencias en nombres de ciudad
ciudad_idx = np.random.choice(len(df), 80, replace=False)
variaciones = ['madrid', 'MADRID', 'Madríd', 'barcelona', 'BARCELONA', 'Barna', 'Vlc']
df.loc[ciudad_idx, 'ciudad'] = np.random.choice(variaciones, 80)

# 6) Emails inválidos
email_idx = np.random.choice(len(df), 40, replace=False)
emails_malos = ['noesunmail', 'sin_arroba.com', '@falta_usuario.com', '', 'espacios en email@mail.com']
df.loc[email_idx, 'email'] = np.random.choice(emails_malos, 40)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset generado: {len(df):,} registros x {len(df.columns)} columnas")
print(f"\n⚠️ Este dataset tiene MUCHOS problemas de calidad intencionados")
df.head(10)

---

## 2. Data Profiling: primera exploración

El **profiling** es el primer paso: entender la estructura, tipos y estadísticas básicas del dataset.

In [ ]:
print("=" * 60)
print("DATA PROFILING - Resumen del dataset")
print("=" * 60)

print(f"\n📊 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"📊 Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n--- Tipos de datos ---")
print(df.dtypes)

print("\n--- Estadísticas numéricas ---")
display(df.describe().round(2))

print("\n--- Cardinalidad (valores únicos por columna) ---")
for col in df.columns:
    print(f"  {col:20s}: {df[col].nunique():>6,} valores únicos")

---

## 3. Detección de problemas de calidad

### 3.1 Valores nulos

In [ ]:
nulos = df.isnull().sum()
pct_nulos = (df.isnull().sum() / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% del total': pct_nulos
}).sort_values('Nulos', ascending=False)

print("=" * 50)
print("ANÁLISIS DE VALORES NULOS")
print("=" * 50)
print(resumen_nulos[resumen_nulos['Nulos'] > 0])
print(f"\nTotal de celdas nulas: {nulos.sum():,} de {df.size:,} ({nulos.sum()/df.size*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cols_con_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]
axes[0].barh(cols_con_nulos.index, cols_con_nulos['% del total'], color='#e74c3c')
axes[0].set_xlabel('% de valores nulos')
axes[0].set_title('Porcentaje de nulos por columna', fontweight='bold')
axes[0].invert_yaxis()

nulos_matrix = df.isnull().astype(int)
axes[1].imshow(nulos_matrix.head(200).T, aspect='auto', cmap='Reds', interpolation='nearest')
axes[1].set_yticks(range(len(df.columns)))
axes[1].set_yticklabels(df.columns)
axes[1].set_xlabel('Registros (primeros 200)')
axes[1].set_title('Mapa de calor de nulos', fontweight='bold')

plt.tight_layout()
plt.show()

### 3.2 Duplicados

In [ ]:
duplicados_exactos = df.duplicated().sum()
duplicados_por_id = df.duplicated(subset=['id_cliente']).sum()

print("=" * 50)
print("ANÁLISIS DE DUPLICADOS")
print("=" * 50)
print(f"Duplicados exactos (toda la fila): {duplicados_exactos:,} ({duplicados_exactos/len(df)*100:.1f}%)")
print(f"Duplicados por id_cliente:         {duplicados_por_id:,} ({duplicados_por_id/len(df)*100:.1f}%)")

print("\n--- Ejemplo de registros duplicados ---")
ids_duplicados = df[df.duplicated(subset=['id_cliente'], keep=False)]['id_cliente'].unique()[:3]
for id_dup in ids_duplicados:
    print(f"\nCliente {id_dup}:")
    display(df[df['id_cliente'] == id_dup][['id_cliente', 'nombre', 'ciudad', 'ingresos']])

### 3.3 Outliers

In [ ]:
def detectar_outliers_iqr(serie):
    """Detecta outliers usando el método IQR."""
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = (serie < limite_inf) | (serie > limite_sup)
    return outliers, limite_inf, limite_sup

columnas_numericas = ['edad', 'ingresos', 'compras_anual', 'satisfaccion']

print("=" * 60)
print("ANÁLISIS DE OUTLIERS (Método IQR)")
print("=" * 60)

fig, axes = plt.subplots(1, len(columnas_numericas), figsize=(16, 4))

for i, col in enumerate(columnas_numericas):
    datos_col = df[col].dropna()
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(datos_col)
    n_outliers = outliers.sum()
    pct = n_outliers / len(datos_col) * 100

    print(f"\n  {col}:")
    print(f"    Rango válido: [{lim_inf:.2f}, {lim_sup:.2f}]")
    print(f"    Outliers: {n_outliers} ({pct:.1f}%)")

    axes[i].boxplot(datos_col, vert=True)
    axes[i].set_title(f'{col}\n({n_outliers} outliers)', fontweight='bold', fontsize=10)

plt.suptitle('Boxplots para detección de outliers', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Inconsistencias

In [ ]:
print("=" * 50)
print("ANÁLISIS DE INCONSISTENCIAS")
print("=" * 50)

print("\n--- Valores únicos en 'ciudad' ---")
print(df['ciudad'].value_counts())

ciudades_validas = {'Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Málaga'}
inconsistentes_ciudad = ~df['ciudad'].isin(ciudades_validas)
print(f"\n⚠️ Ciudades inconsistentes: {inconsistentes_ciudad.sum()} registros")

import re
patron_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
emails_no_nulos = df['email'].dropna()
emails_invalidos = ~emails_no_nulos.str.match(patron_email, na=False)
print(f"\n⚠️ Emails inválidos: {emails_invalidos.sum()} de {len(emails_no_nulos)} ({emails_invalidos.sum()/len(emails_no_nulos)*100:.1f}%)")

edades_no_nulas = df['edad'].dropna()
edades_invalidas = (edades_no_nulas < 0) | (edades_no_nulas > 120)
print(f"⚠️ Edades fuera de rango [0-120]: {edades_invalidas.sum()} registros")

ingresos_no_nulos = df['ingresos'].dropna()
ingresos_negativos = (ingresos_no_nulos < 0).sum()
print(f"⚠️ Ingresos negativos: {ingresos_negativos} registros")

---

## 4. Quality Score

Calculamos una puntuación global de calidad del dataset (0-100).

In [ ]:
def calcular_quality_score(dataframe):
    """Calcula un quality score de 0 a 100 para un DataFrame."""
    n = len(dataframe)

    # Completitud: % de celdas no nulas
    completitud = (1 - dataframe.isnull().sum().sum() / dataframe.size) * 100

    # Unicidad: % de filas únicas
    unicidad = (1 - dataframe.duplicated().sum() / n) * 100

    # Validez: % de registros con valores dentro de rangos válidos
    validez_checks = []
    if 'edad' in dataframe.columns:
        edad_valida = dataframe['edad'].dropna().between(0, 120).mean() * 100
        validez_checks.append(edad_valida)
    if 'ingresos' in dataframe.columns:
        ingreso_valido = (dataframe['ingresos'].dropna() >= 0).mean() * 100
        validez_checks.append(ingreso_valido)
    if 'email' in dataframe.columns:
        patron = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        email_valido = dataframe['email'].dropna().str.match(patron).mean() * 100
        validez_checks.append(email_valido)
    validez = np.mean(validez_checks) if validez_checks else 100

    # Consistencia: % de ciudades con formato estándar
    if 'ciudad' in dataframe.columns:
        ciudades_std = {'Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Málaga'}
        consistencia = dataframe['ciudad'].isin(ciudades_std).mean() * 100
    else:
        consistencia = 100

    # Score ponderado
    score = 0.30 * completitud + 0.25 * unicidad + 0.25 * validez + 0.20 * consistencia

    return {
        'completitud': round(completitud, 2),
        'unicidad': round(unicidad, 2),
        'validez': round(validez, 2),
        'consistencia': round(consistencia, 2),
        'score_global': round(score, 2)
    }

score_antes = calcular_quality_score(df)

print("=" * 50)
print("QUALITY SCORE - ANTES de limpieza")
print("=" * 50)
for metrica, valor in score_antes.items():
    barra = '█' * int(valor // 5) + '░' * (20 - int(valor // 5))
    print(f"  {metrica:15s}: {barra} {valor:.1f}%")

nivel = 'Excelente' if score_antes['score_global'] >= 90 else \
        'Buena' if score_antes['score_global'] >= 75 else \
        'Aceptable' if score_antes['score_global'] >= 50 else 'Pobre'
print(f"\n📊 Calificación: {nivel}")

---

## 5. Limpieza de datos

Ahora aplicamos las técnicas de limpieza para mejorar la calidad.

In [ ]:
df_limpio = df.copy()
print(f"Registros antes de limpieza: {len(df_limpio):,}")

# 1) Eliminar duplicados exactos
df_limpio = df_limpio.drop_duplicates()
print(f"Tras eliminar duplicados:    {len(df_limpio):,}")

# 2) Normalizar ciudades
mapa_ciudades = {
    'madrid': 'Madrid', 'MADRID': 'Madrid', 'Madríd': 'Madrid',
    'barcelona': 'Barcelona', 'BARCELONA': 'Barcelona', 'Barna': 'Barcelona',
    'Vlc': 'Valencia'
}
df_limpio['ciudad'] = df_limpio['ciudad'].replace(mapa_ciudades)

# 3) Corregir edades inválidas → NaN, luego imputar con mediana
df_limpio.loc[(df_limpio['edad'] < 0) | (df_limpio['edad'] > 120), 'edad'] = np.nan
mediana_edad = df_limpio['edad'].median()
df_limpio['edad'] = df_limpio['edad'].fillna(mediana_edad)

# 4) Corregir ingresos negativos → NaN, luego imputar con mediana
df_limpio.loc[df_limpio['ingresos'] < 0, 'ingresos'] = np.nan
# Eliminar outliers extremos (>3 desviaciones estándar)
media_ing = df_limpio['ingresos'].mean()
std_ing = df_limpio['ingresos'].std()
df_limpio.loc[df_limpio['ingresos'] > media_ing + 3 * std_ing, 'ingresos'] = np.nan
mediana_ingresos = df_limpio['ingresos'].median()
df_limpio['ingresos'] = df_limpio['ingresos'].fillna(mediana_ingresos)

# 5) Imputar nulos restantes
df_limpio['compras_anual'] = df_limpio['compras_anual'].fillna(df_limpio['compras_anual'].median())
df_limpio['satisfaccion'] = df_limpio['satisfaccion'].fillna(df_limpio['satisfaccion'].median())

# 6) Emails inválidos → NaN (no podemos inventar un email)
patron_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
emails_malos = ~df_limpio['email'].str.match(patron_email, na=True)
df_limpio.loc[emails_malos, 'email'] = np.nan

print(f"Registros tras limpieza:     {len(df_limpio):,}")
print("\n✅ Limpieza completada")

---

## 6. Verificación: comparar ANTES vs DESPUÉS

In [ ]:
score_despues = calcular_quality_score(df_limpio)

print("=" * 60)
print("COMPARACIÓN: ANTES vs DESPUÉS de limpieza")
print("=" * 60)
print(f"\n{'Métrica':<18} {'Antes':>10} {'Después':>10} {'Mejora':>10}")
print("-" * 50)
for metrica in score_antes:
    antes = score_antes[metrica]
    despues = score_despues[metrica]
    mejora = despues - antes
    emoji = '📈' if mejora > 0 else '➡️'
    print(f"{emoji} {metrica:<16} {antes:>9.1f}% {despues:>9.1f}% {mejora:>+9.1f}%")

# Gráfico comparativo
metricas = list(score_antes.keys())
valores_antes = [score_antes[m] for m in metricas]
valores_despues = [score_despues[m] for m in metricas]

x = np.arange(len(metricas))
ancho = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - ancho/2, valores_antes, ancho, label='Antes', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + ancho/2, valores_despues, ancho, label='Después', color='#2ecc71', alpha=0.8)

ax.set_ylabel('Puntuación (%)')
ax.set_title('Quality Score: Antes vs Después de la limpieza', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metricas, rotation=15)
ax.legend()
ax.set_ylim(0, 110)
ax.axhline(y=75, color='orange', linestyle='--', alpha=0.5, label='Umbral "Buena calidad"')
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---

## 7. Resumen y conclusiones

### Lo que hemos aprendido:

| Problema | Técnica de detección | Técnica de limpieza |
|----------|---------------------|---------------------|
| Valores nulos | `isnull().sum()` | Imputación (media/mediana/moda), eliminación |
| Duplicados | `duplicated()` | `drop_duplicates()` |
| Outliers | IQR, z-score, boxplot | Reemplazar por NaN + imputar, o eliminar |
| Inconsistencias | `value_counts()`, regex | Normalización, mapeos de corrección |

### Regla de oro:
> **Siempre mide la calidad ANTES y DESPUÉS de cualquier proceso de limpieza.**
> Un quality score te da una visión objetiva de la mejora.

---

## Ejercicios para practicar

1. Añade una dimensión más al quality score: **oportunidad** (basándote en la fecha de registro).
2. Implementa detección de outliers con z-score y compara con el método IQR.
3. Descarga un dataset real de Kaggle y ejecuta este pipeline completo sobre él.